In [1]:
# I import pandas to load, explore, and manipulate the dataset.
import pandas as pd

# I import train_test_split to split the dataset into training, validation, and test sets.
from sklearn.model_selection import train_test_split

# I import mutual_info_score to measure the relationship between categorical features and the target variable.
from sklearn.metrics import mutual_info_score

# I import DictVectorizer to convert categorical features into numerical features using one-hot encoding.
from sklearn.feature_extraction import DictVectorizer

# I import LogisticRegression to build the classification model.
from sklearn.linear_model import LogisticRegression

# I import accuracy_score to evaluate the prediction accuracy of the model.
from sklearn.metrics import accuracy_score

In [2]:
# I define the URL of the 2026 lead-scoring dataset provided for this homework.
data_url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

# I load the CSV file directly from the URL into a pandas DataFrame.
df = pd.read_csv(data_url)

# I display the first five rows to make sure that the dataset was loaded correctly.
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# I check the number of rows and columns in the dataset.
df.shape

(5000, 9)

In [4]:
# I inspect the dataset structure, column names, data types, and non-null values.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [5]:
# I display all column names to understand which features are available in the dataset.
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [6]:
# I check how many missing values are present in each column before applying any transformations.
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [8]:
# I define the categorical features so that I can handle their missing values separately.
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

# I define the numerical features so that I can handle their missing values separately.
numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

In [9]:
# I replace missing values in categorical features with 'NA' as required by the homework.
df[categorical] = df[categorical].fillna('NA')

# I replace missing values in numerical features with 0.0 as required by the homework.
df[numerical] = df[numerical].fillna(0.0)

In [10]:
# I check the missing values again to verify that all missing values have been filled.
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


In [11]:
# I count the occurrences of each value in the industry column to examine its frequency distribution.
df['industry'].value_counts()

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463
NA,240


In [12]:
# I calculate the mode of the industry column to find its most frequent value.
industry_mode = df['industry'].mode()[0]

# I display the most frequent value in the industry column.
print("Most frequent industry:", industry_mode)

Most frequent industry: technology


In [13]:
# I calculate the correlation matrix for the numerical features only.
correlation_matrix = df[numerical].corr()

# I display the correlation matrix to compare the relationships between numerical features.
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


| Feature pair | Correlation |
|---|---:|
| `interaction_count` & `lead_score` | **0.915746** |
| `number_of_courses_viewed` & `lead_score` | 0.757204 |
| `number_of_courses_viewed` & `interaction_count` | 0.721609 |
| `annual_income` & `interaction_count` | 0.122842 |

In [14]:
# I split the dataset into full training and test sets using the exact parameters required by the homework.
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# I split the full training set again to create separate training and validation sets.
df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

In [15]:
# I check the sizes of the training, validation, and test sets.
print("Training set:", len(df_train))
print("Validation set:", len(df_val))
print("Test set:", len(df_test))

Training set: 3000
Validation set: 1000
Test set: 1000


In [16]:
# I extract the converted column as the target variable for each dataset.
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [17]:
# I remove the target variable from the feature DataFrames to prevent data leakage.
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [18]:
# I verify that the target variable is no longer included in the training features.
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='object')

In [19]:
# I create a function to calculate the mutual information between a categorical feature and the target variable.
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [20]:
# I calculate the mutual information score for each categorical feature using the training set only.
mi_scores = df_train[categorical].apply(mutual_info_converted_score)

# I sort the scores from highest to lowest to identify the most informative categorical feature.
mi_scores = mi_scores.sort_values(ascending=False)

# I display the mutual information scores.
mi_scores

,0
lead_source,0.034346
employment_status,0.018966
industry,0.001995
location,0.001043


In [21]:
# I round each mutual information score to 2 decimal places as required by the homework.
mi_scores_rounded = mi_scores.apply(lambda score: round(score, 2))

# I display the rounded mutual information scores.
mi_scores_rounded

,0
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


In [22]:
# I convert the training DataFrame into a list of dictionaries for DictVectorizer.
train_dicts = df_train.to_dict(orient='records')

In [23]:
# I create a DictVectorizer to convert categorical features into one-hot encoded numerical features.
dv = DictVectorizer(sparse=False)

# I fit the vectorizer on the training data and transform the features into a numerical matrix.
X_train = dv.fit_transform(train_dicts)

In [24]:
# I check the shape of the transformed training feature matrix.
X_train.shape

(3000, 28)

In [25]:
# I convert the validation DataFrame into a list of dictionaries.
val_dicts = df_val.to_dict(orient='records')

# I transform the validation data using the vectorizer fitted on the training data.
X_val = dv.transform(val_dicts)

In [26]:
# I check the shape of the transformed validation feature matrix.
X_val.shape

(1000, 28)

In [27]:
# I create the logistic regression model using the exact parameters required by the homework.
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

# I train the logistic regression model using the training data.
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [28]:
# I calculate the probability that each validation observation belongs to the positive class.
y_pred_proba = model.predict_proba(X_val)[:, 1]

# I display the first five predicted probabilities.
y_pred_proba[:5]

array([0.61725691, 0.44364108, 0.60433996, 0.64562246, 0.52214159])

In [29]:
# I convert the predicted probabilities into binary predictions using a threshold of 0.5.
y_pred = (y_pred_proba >= 0.5).astype(int)

In [30]:
# I calculate the accuracy of the model on the validation dataset.
accuracy = accuracy_score(y_val, y_pred)

# I display the original validation accuracy.
print("Validation accuracy:", accuracy)

# I round the accuracy to 2 decimal places as required by the homework.
print("Rounded validation accuracy:", round(accuracy, 2))

Validation accuracy: 0.645
Rounded validation accuracy: 0.65


In [31]:
# I save the original validation accuracy from Question 4 for comparison.
original_accuracy = accuracy

# I display the original accuracy without rounding.
print("Original accuracy:", original_accuracy)

Original accuracy: 0.645


In [32]:
# I define the features that I need to evaluate for feature elimination.
features_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [33]:
# I create an empty dictionary to store the accuracy difference for each removed feature.
feature_differences = {}

# I evaluate each feature by removing it and retraining the model.
for feature in features_to_test:

    # I create copies of the training and validation data without the current feature.
    df_train_reduced = df_train.drop(columns=[feature])
    df_val_reduced = df_val.drop(columns=[feature])

    # I convert the reduced DataFrames into dictionaries for one-hot encoding.
    train_dicts_reduced = df_train_reduced.to_dict(orient='records')
    val_dicts_reduced = df_val_reduced.to_dict(orient='records')

    # I create a new DictVectorizer for the reduced feature set.
    dv_reduced = DictVectorizer(sparse=False)

    # I fit the vectorizer on the reduced training data.
    X_train_reduced = dv_reduced.fit_transform(train_dicts_reduced)

    # I transform the reduced validation data using the same vectorizer.
    X_val_reduced = dv_reduced.transform(val_dicts_reduced)

    # I create a new logistic regression model using the same parameters as in Question 4.
    model_reduced = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    # I train the model without the current feature.
    model_reduced.fit(X_train_reduced, y_train)

    # I make predictions on the validation set.
    y_pred_reduced = model_reduced.predict(X_val_reduced)

    # I calculate the validation accuracy without the current feature.
    accuracy_reduced = accuracy_score(y_val, y_pred_reduced)

    # I calculate the difference between the original accuracy and the reduced-model accuracy.
    difference = original_accuracy - accuracy_reduced

    # I store the result for the current feature.
    feature_differences[feature] = difference

    # I display the accuracy and difference for the current feature.
    print(
        f"{feature}: accuracy = {accuracy_reduced:.6f}, "
        f"difference = {difference:.6f}"
    )

lead_source: accuracy = 0.642000, difference = 0.003000
number_of_courses_viewed: accuracy = 0.643000, difference = 0.002000
interaction_count: accuracy = 0.601000, difference = 0.044000


In [34]:
# I identify the feature with the smallest difference in validation accuracy.
least_useful_feature = min(feature_differences, key=feature_differences.get)

# I display the feature with the smallest difference.
print("Feature with the smallest difference:", least_useful_feature)

Feature with the smallest difference: number_of_courses_viewed


In [35]:
# I define the C values that I need to test for regularized logistic regression.
C_values = [0.000001, 0.00001, 0.0001, 0.001]

In [36]:
# I create an empty dictionary to store the validation accuracy for each C value.
c_accuracies = {}

# I train and evaluate a logistic regression model for each C value.
for C in C_values:

    # I create a logistic regression model with the current C value.
    model_c = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    # I train the model using all training features.
    model_c.fit(X_train, y_train)

    # I make predictions on the validation dataset.
    y_pred_c = model_c.predict(X_val)

    # I calculate the validation accuracy.
    accuracy_c = accuracy_score(y_val, y_pred_c)

    # I round the accuracy to 3 decimal places as required by the homework.
    rounded_accuracy = round(accuracy_c, 3)

    # I store the rounded accuracy for the current C value.
    c_accuracies[C] = rounded_accuracy

    # I display the C value and its validation accuracy.
    print(f"C = {C}: accuracy = {rounded_accuracy}")

C = 1e-06: accuracy = 0.598
C = 1e-05: accuracy = 0.598
C = 0.0001: accuracy = 0.613
C = 0.001: accuracy = 0.645


In [37]:
# I find the highest rounded validation accuracy among all tested C values.
best_accuracy = max(c_accuracies.values())

# I find all C values that achieve the highest rounded accuracy.
best_C_values = [
    C for C, acc in c_accuracies.items()
    if acc == best_accuracy
]

# If multiple C values have the same best accuracy, I select the smallest C as required.
best_C = min(best_C_values)

# I display the best C value and its validation accuracy.
print("Best C:", best_C)
print("Best validation accuracy:", best_accuracy)

Best C: 0.001
Best validation accuracy: 0.645


In [38]:
# I summarize my final answers for Homework 3.
final_answers = {
    "Question 1": "technology",
    "Question 2": "interaction_count and lead_score",
    "Question 3": "lead_source",
    "Question 4": 0.65,
    "Question 5": "number_of_courses_viewed",
    "Question 6": 0.001
}

# I display all final answers in a clear format.
for question, answer in final_answers.items():
    print(f"{question}: {answer}")

Question 1: technology
Question 2: interaction_count and lead_score
Question 3: lead_source
Question 4: 0.65
Question 5: number_of_courses_viewed
Question 6: 0.001
